# Customs Delay Risk Predictor — Training Notebook
**Opsi A: Cross-Border Trade & Customs Delay Dataset**

Melatih 2 model:
- `model_delay` → XGBoost Regressor → prediksi `Customs_Delay_Days`
- `model_risk`  → XGBoost Classifier → prediksi `Risk_Flag` (0=Low, 1=High)

**Jalankan di Google Colab: Runtime → Run All**

## Step 1 — Install & Import

In [ ]:
!pip install xgboost shap --quiet

import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import pickle
import json
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    mean_absolute_error, r2_score,
    classification_report, accuracy_score, f1_score
)

print('Imports OK')
print(f'XGBoost: {xgb.__version__}')

## Step 2 — Upload & Load Dataset

Upload file `trade_customs_dataset.csv` ke Colab.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Pilih: trade_customs_dataset.csv

In [ ]:
df = pd.read_csv('trade_customs_dataset.csv')
print(f'Shape: {df.shape}')
print(f'Kolom: {df.columns.tolist()}')
df.head(3)

## Step 3 — EDA Singkat

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())

print('\n=== Target: Customs_Delay_Days ===')
print(df['Customs_Delay_Days'].describe())

print('\n=== Target: Risk_Flag ===')
print(df['Risk_Flag'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['Customs_Delay_Days'].hist(bins=20, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribusi Customs_Delay_Days')
df['Risk_Flag'].value_counts().plot(kind='bar', ax=axes[1], color=['#e74c3c','#2ecc71'], edgecolor='white')
axes[1].set_title('Risk_Flag (0=Low, 1=High)')
axes[1].set_xticklabels(['High Risk (1)', 'Low Risk (0)'], rotation=0)
plt.tight_layout()
plt.show()

## Step 4 — Preprocessing

In [ ]:
CATEGORICAL_COLS = [
    'Origin_Country', 'Destination_Country', 'Transport_Mode',
    'Carrier_Name', 'Commodity_Type', 'Document_Status',
    'Tariff_Category', 'Inspection_Type',
]

NUMERICAL_COLS = [
    'Declared_Value_USD', 'Weight_kg', 'Compliance_Score',
    'Prior_Offense_Count', 'Route_Risk_Index',
]

FEATURE_COLS = CATEGORICAL_COLS + NUMERICAL_COLS
TARGET_REG   = 'Customs_Delay_Days'
TARGET_CLF   = 'Risk_Flag'

# Bersihkan missing values
df_clean = df[FEATURE_COLS + [TARGET_REG, TARGET_CLF]].dropna()
print(f'Rows setelah dropna: {len(df_clean)}')

# Label encode semua kolom kategorikal
label_encoders = {}
df_encoded = df_clean.copy()
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    label_encoders[col] = le
    print(f'  {col}: {len(le.classes_)} unique values')

print('Encoding selesai')

In [ ]:
X = df_encoded[FEATURE_COLS]
y_reg = df_encoded[TARGET_REG]
y_clf = df_encoded[TARGET_CLF]

X_train, X_test, y_reg_train, y_reg_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)
_, _, y_clf_train, y_clf_test = train_test_split(X, y_clf, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## Step 5 — Training Model

**Kenapa XGBoost?**
- Terbaik untuk tabular/CSV data
- Akurasi tinggi tanpa GPU
- Native SHAP support (explainability)
- Training < 1 menit untuk 10k rows
- Format `.json` portabel untuk backend

In [ ]:
# Model 1: Regressor — prediksi jumlah hari delay
print('Training XGBoost Regressor...')
model_delay = xgb.XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, tree_method='hist', verbosity=0
)
model_delay.fit(X_train, y_reg_train, eval_set=[(X_test, y_reg_test)], verbose=100)

y_pred_reg = model_delay.predict(X_test)
mae = mean_absolute_error(y_reg_test, y_pred_reg)
r2  = r2_score(y_reg_test, y_pred_reg)
print(f'MAE  : {mae:.3f} hari  (target < 2.0)')
print(f'R2   : {r2:.3f}       (target > 0.70)')

In [ ]:
# Model 2: Classifier — prediksi Risk_Flag
print('Training XGBoost Classifier...')
neg = (y_clf_train == 0).sum()
pos = (y_clf_train == 1).sum()
scale_pos_weight = neg / pos
print(f'  Class weight (neg/pos): {scale_pos_weight:.2f}')

model_risk = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42, tree_method='hist', verbosity=0, eval_metric='logloss'
)
model_risk.fit(X_train, y_clf_train, eval_set=[(X_test, y_clf_test)], verbose=100)

y_pred_clf = model_risk.predict(X_test)
acc = accuracy_score(y_clf_test, y_pred_clf)
f1  = f1_score(y_clf_test, y_pred_clf)
print(f'Accuracy : {acc:.3f}  (target > 0.70)')
print(f'F1 Score : {f1:.3f}  (target > 0.70)')
print(classification_report(y_clf_test, y_pred_clf, target_names=['Low Risk','High Risk']))

## Step 6 — SHAP Explainer

In [ ]:
print('Computing SHAP...')
explainer = shap.TreeExplainer(model_delay)

# Visualisasi feature importance global
shap_vals = explainer.shap_values(X_test.iloc[:200])
shap.summary_plot(shap_vals, X_test.iloc[:200], feature_names=FEATURE_COLS, plot_type='bar')

In [ ]:
# Test prediksi + SHAP untuk 1 sample (simulasi request API)
sample = X_test.iloc[[0]]
shap_single = explainer.shap_values(sample)[0]
shap_dict   = dict(zip(FEATURE_COLS, shap_single))
shap_top5   = sorted(shap_dict.items(), key=lambda x: abs(x[1]), reverse=True)[:5]
total       = sum(abs(v) for _, v in shap_top5)
shap_pct    = {k: round(abs(v)/total*100, 1) for k, v in shap_top5}

pred_delay = float(model_delay.predict(sample)[0])
pred_risk  = int(model_risk.predict(sample)[0])
pred_prob  = float(model_risk.predict_proba(sample)[0][1])

print('--- Contoh Output Backend ---')
print(f'customs_delay_days : {pred_delay:.1f} hari')
print(f'risk_level         : {"HIGH" if pred_risk==1 else "LOW"}')
print(f'confidence         : {pred_prob:.0%}')
print(f'shap_breakdown     : {shap_pct}')

## Step 7 — Simpan & Download Artefak Model

In [ ]:
import os, shutil
os.makedirs('models', exist_ok=True)

model_delay.save_model('models/model_delay.json')
model_risk.save_model('models/model_risk.json')
print('Saved: model_delay.json, model_risk.json')

with open('models/explainer.pkl', 'wb') as f:
    pickle.dump(explainer, f)
print('Saved: explainer.pkl')

with open('models/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)
print('Saved: label_encoders.pkl')

with open('models/feature_columns.json', 'w') as f:
    json.dump({'feature_cols': FEATURE_COLS, 'categorical_cols': CATEGORICAL_COLS, 'numerical_cols': NUMERICAL_COLS}, f, indent=2)
print('Saved: feature_columns.json')

encoder_classes = {col: list(le.classes_) for col, le in label_encoders.items()}
with open('models/encoder_classes.json', 'w') as f:
    json.dump(encoder_classes, f, indent=2)
print('Saved: encoder_classes.json')

# Zip semua dan download
shutil.make_archive('models_export', 'zip', 'models')
from google.colab import files
files.download('models_export.zip')
print('Download models_export.zip → ekstrak ke backend/models/')

## Step 8 — Ringkasan Metrics

In [ ]:
print('=' * 50)
print('FINAL MODEL SUMMARY')
print('=' * 50)
print(f'Dataset  : trade_customs_dataset.csv ({len(df_clean):,} rows, {len(FEATURE_COLS)} features)')
print(f'Split    : 80% train / 20% test')
print()
print(f'[Regressor — Customs Delay Days]')
print(f'  MAE  = {mae:.3f} hari  (target < 2.0)  {"PASS" if mae < 2.0 else "FAIL"}')
print(f'  R2   = {r2:.3f}        (target > 0.70)  {"PASS" if r2 > 0.70 else "FAIL"}')
print()
print(f'[Classifier — Risk Flag]')
print(f'  Accuracy = {acc:.3f}  (target > 0.70)  {"PASS" if acc > 0.70 else "FAIL"}')
print(f'  F1       = {f1:.3f}  (target > 0.70)  {"PASS" if f1 > 0.70 else "FAIL"}')
print()
print('Output files (ekstrak ke backend/models/):')
for fname in ['model_delay.json','model_risk.json','explainer.pkl','label_encoders.pkl','feature_columns.json','encoder_classes.json']:
    size = os.path.getsize(f'models/{fname}')
    print(f'  {fname} ({size/1024:.1f} KB)')